##Current Decision:

Focusing on WCT (white line) calculations; `wct_threshold = .05` and `wct_width = 300` have the best-looking results.
___
This notebook now runs everything needed to process and visualize data on its own (first cell to install atmolidarpy must be run once per session first) - it was an already-existing example code file from the lidarpy module that required minimal modification by gemini to run in the colab environment.

Modify line 18 to customize which date's data is being imported.

Quicklook files are pngs and can be viewed directly. The rest are .nc data files that can be viewed with the cell at the bottom.

Parameters to be tinkered with are in `parse_args()`, specifically those on lines 270-273. (Formatting from lines 128-136 may be more intuitive for reference). Changing line 294 changes first term in quicklook file name.

In [ ]:
!pip install atmolidarpy
import lidarpy

In [ ]:
import argparse
from pathlib import Path
import nest_asyncio; nest_asyncio.apply()

from cloudnet_api_client import APIClient
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from lidarpy.retrieval.ablh import detect_ablh


#REPO_ROOT = Path(__file__).resolve().parents[1]
DEFAULT_OUTPUT_DIR = Path.cwd() #REPO_ROOT / "artifacts" / "cloudnet_chm15k_abl_workbench"
METHODS = ("wct", "temporal_variance")
CLOUDNET_SITE = "granada"
CLOUDNET_DATE = "2024-02-08"
CLOUDNET_PRODUCT = "lidar"
CLOUDNET_FILENAME = f"{''.join(CLOUDNET_DATE.split('-'))}_granada_chm15k_77a75f3b.nc"#_Granada_CHM170119_000.nc" #_granada_chm15k_77a75f3b.nc"
CLOUDNET_INPUT = Path(f'/content/{CLOUDNET_FILENAME}') # Modified filename for consistency

RAW_FILENAME = f"{''.join(CLOUDNET_DATE.split('-'))}_Granada_CHM170119_000.nc"#_Granada_CHM170119_000.nc" #_granada_chm15k_77a75f3b.nc"
RAW_INPUT = Path(f'/content/{RAW_FILENAME}')

CLOUDNET_INPUT.exists(), RAW_INPUT.exists()

In [ ]:
dataset = load_cloudnet_for_ablh(input_path=RAW_INPUT, variable='beta_raw', time_frequency=None)

In [ ]:
def ensure_cloudnet_input(
    input_path: Path,
    *,
    download: bool,
) -> Path:
    """Return an existing Cloudnet file, downloading the default fixture if needed."""
    input_path = input_path.expanduser().resolve()
    if input_path.exists():
        print(f"File already exists: {input_path}")
        return input_path
    if not download:
        raise FileNotFoundError(
            f"Cloudnet input file not found: {input_path}. "
            "Re-run without --no-download or pass --input to an existing file."
        )
    if input_path.name != CLOUDNET_FILENAME:
        raise FileNotFoundError(
            f"Cloudnet input file not found: {input_path}. "
            "Automatic download is only configured for the default Granada CHM15k file."
        )
    print(f"Downloading {CLOUDNET_FILENAME}...")
    return download_cloudnet_lidar_product(target_path)


def download_cloudnet_lidar_product(target_path: Path) -> Path:
    """Download the default Granada CHM15k lidar product with Cloudnet API client."""
    client = APIClient()
    files = client.files(
        site_id=CLOUDNET_SITE,
        date=CLOUDNET_DATE,
        product_id=CLOUDNET_PRODUCT,
    )
    matching_files = [file for file in files if file.filename == CLOUDNET_FILENAME]
    if not matching_files:
        raise FileNotFoundError(
            f"Cloudnet did not return {CLOUDNET_FILENAME} for "
            f"site={CLOUDNET_SITE}, date={CLOUDNET_DATE}, product={CLOUDNET_PRODUCT}"
        )

    target_path.parent.mkdir(parents=True, exist_ok=True)
    downloaded_paths = client.download(
        matching_files[0],
        output_directory=target_path.parent,
        progress=True,
        validate_checksum=True,
    )
    downloaded_path = Path(downloaded_paths[0])
    if downloaded_path != target_path:
        downloaded_path.replace(target_path)
    return target_path


def run_ceilometer_ablh_workbench(
    cloudnet_path: Path,
    raw_path: Path,
    output_dir: Path,
    *,
    variable: str = "beta_smooth",
    time_frequency: str | None = "5min",
    min_range: float = 500.0,
    max_range: float = 4000.0,
    wct_threshold: float = 0.02,
    wct_width: float = 300.0,
    temporal_threshold: float = 1e-5,
    temporal_window_minutes: float = 10.0,
    snr_threshold = 0.01,
    time_sigma = 1,
    range_sigma = 3,
    parameter = 1
) -> tuple[dict[str, Path], Path]:
    """Run WCT and temporal-variance ABLH detection on Cloudnet ceilometer data."""
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load the processed dataset (which will contain 'beta_raw' if variable='beta_raw')
    """"
    dataset = load_cloudnet_for_ablh(
        input_path,
        variable=variable,
        time_frequency=time_frequency,
    )
    """

    dataset = load_cloudnet_for_ablh(input_path=raw_path, variable='beta_raw', time_frequency=None)
    cloudnet_dataset = load_cloudnet_for_ablh(input_path=cloudnet_path, variable='beta', time_frequency=None)

    print("Dataset loaded successfully (after initial load_cloudnet_for_ablh).") # Clarified debug print

    # Implement NaN mask from original cdataset['beta']
    # Load original again to get 'beta' variable with its original dimensions before load_cloudnet_for_ablh processes it.
    original_nan_mask = np.isnan(cloudnet_dataset['beta'])

    # Reindex the original NaN mask to match the coordinates of the processed dataset[variable]
    nan_mask_reindexed = original_nan_mask.reindex_like(dataset[variable], method='nearest')

    # Apply the reindexed NaN mask to filter dataset[variable]
    dataset[variable] = dataset[variable].where(~nan_mask_reindexed)

    if dataset[variable].isnull().all():
        print(f"WARNING: After applying cdataset['beta'] NaN mask, '{variable}' is entirely NaN. This might cause issues downstream.")
    else:
        print(f"'{variable}' masked by cdataset['beta'] NaN mask. Valid data points remain.")

    # --- Removed SNR filtering section entirely as per user's request ---

    frequency_label = time_frequency or "native"
    range_label = f"r{min_range:g}-{max_range:g}m"
    output_paths: dict[str, Path] = {}
    results: dict[str, xr.Dataset] = {}

    method_options = {
        "wct": {
            "threshold": wct_threshold,
            "wct_width": wct_width,
            "time_window_minutes": temporal_window_minutes,
        },
        "temporal_variance": {
            "threshold": temporal_threshold,
            "wct_width": wct_width,
            "time_window_minutes":temporal_window_minutes
        },
    }

    print("Starting ABLH detection loop.") # Debug print
    for method in METHODS:
        # Check if dataset[variable] has any finite values before calling detect_ablh
        if not np.isfinite(dataset[variable]).any():
            print(f"Skipping {method} method: '{variable}' has no finite values after filtering.")
            continue # Skip this method if no valid data

        # Original code calls detect_ablh three times. I will just store the result once and reuse.
        ablh_result = detect_ablh(
            dataset,
            input_kind="cloudnet",
            variable=variable,
            method=method,
            min_range=min_range,
            max_range=max_range,
            **method_options[method],
        )
        ablh_result.attrs.update(
            {
                "source_cloudnet_file": str(input_path),
                "cloudnet_variable": variable,
                "time_frequency": frequency_label,
                "wct_threshold": float(wct_threshold),
                "wct_width_m": float(wct_width),
                "temporal_threshold": float(temporal_threshold),
                "temporal_window_minutes": float(temporal_window_minutes),
            }
        )
        output_path = (
            output_dir
            / f"{parameter}_{method}_{input_path.stem}_{variable}_{frequency_label}_{range_label}_ablh.nc"
        )
        ablh_result.to_netcdf(output_path)
        output_paths[method] = output_path
        results[method] = ablh_result
        print(f"ABLH detection completed for method: {method}") # Debug print

    quicklook_path = (
        output_dir
        / f"quicklook_{parameter}_{input_path.stem}_{variable}_{frequency_label}_{range_label}_ablh.png"
    )
    print("Attempting to plot quicklook.") # Debug print
    try:
        plot_cloudnet_ablh_quicklook(
            cloudnet=dataset,
            ablh_results=results,
            variable=variable,
            min_range=min_range,
            max_range=max_range,
            output_path=quicklook_path,
        )
        print("Quicklook plotted successfully.") # Debug print
    except ValueError as e:
        print(f"ERROR: Failed to plot quicklook: {e}")
        # If plotting fails, we should still return the ablh_paths if they were created
        # Or, decide to raise the error further if quicklook is mandatory.
        # For now, just print and continue to return what's available.

    print("run_ceilometer_ablh_workbench finished.") # Debug print
    return output_paths, quicklook_path


def plot_cloudnet_ablh_quicklook(
    *,
    cloudnet: xr.Dataset,
    ablh_results: dict[str, xr.Dataset],
    variable: str,
    min_range: float,
    max_range: float,
    output_path: Path,
) -> None:
    """Plot Cloudnet signal and overlaid ABLH from all requested methods."""
    backscatter = cloudnet[variable].transpose("time", "range")
    ranges = backscatter["range"].values.astype(float)
    range_mask = (ranges >= min_range) & (ranges <= max_range)
    times = backscatter["time"].values
    plot_signal = _log10_positive(backscatter.values[:, range_mask])

    fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)
    mesh = ax.pcolormesh(
        times,
        ranges[range_mask] / 1000.0,
        plot_signal.T,
        shading="auto",
        cmap="viridis",
    )
    colors = {"wct": "white", "temporal_variance": "tab:red"}
    labels = {
        "wct": "ABLH WCT",
        "temporal_variance": "ABLH temporal variance",
    }
    for method, result in ablh_results.items():
        ax.plot(
            result["time"].values,
            result["ablh"].values / 1000.0,
            color=colors.get(method, "tab:orange"),
            lw=1.6 if method == "temporal_variance" else 2.0,
            label=labels.get(method, method),
        )

    title = cloudnet.attrs.get("title", "Cloudnet ceilometer")
    ax.set_title(f"{title}: ABLH from {variable}")
    ax.set_xlabel("UTC time")
    ax.set_ylabel("Range above instrument [km]")
    ax.set_ylim(min_range / 1000.0, max_range / 1000.0)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper right")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    cbar = fig.colorbar(mesh, ax=ax)
    cbar.set_label(f"log10({variable})")
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def _log10_positive(values: np.ndarray) -> np.ndarray:
    # Replace non-positive values with NaN. Original NaNs are preserved.
    processed_values = np.where(values > 0, values, np.nan)

    # Identify finite positive values to calculate the floor.
    finite_positive_mask = np.isfinite(processed_values)

    if not finite_positive_mask.any():
        # If there are no finite positive values, return an array of NaNs.
        # This will result in a blank area on the plot.
        return np.full_like(values, np.nan, dtype=float)

    # Calculate the 1st percentile from the *finite positive* values.
    floor = np.nanpercentile(processed_values[finite_positive_mask], 1)

    # Ensure the floor is positive and finite.
    if not np.isfinite(floor) or floor <= 0:
        floor = np.nanmin(processed_values[finite_positive_mask])

    # Apply the floor only to values that are valid but less than the floor,
    # ensuring NaNs are preserved as NaNs by np.maximum behavior.
    final_values = np.maximum(processed_values, floor)

    return np.log10(final_values)

In [ ]:
def ensure_cloudnet_input(
    input_path: Path,
    *,
    download: bool,
) -> Path:
    """Return an existing Cloudnet file, downloading the default fixture if needed."""
    input_path = input_path.expanduser().resolve()
    if input_path.exists():
        print(f"File already exists: {input_path}")
        return input_path
    if not download:
        raise FileNotFoundError(
            f"Cloudnet input file not found: {input_path}. "
            "Re-run without --no-download or pass --input to an existing file."
        )
    if input_path.name != CLOUDNET_FILENAME:
        raise FileNotFoundError(
            f"Cloudnet input file not found: {input_path}. "
            "Automatic download is only configured for the default Granada CHM15k file."
        )
    print(f"Downloading {CLOUDNET_FILENAME}...")
    return download_cloudnet_lidar_product(target_path)


def download_cloudnet_lidar_product(target_path: Path) -> Path:
    """Download the default Granada CHM15k lidar product with Cloudnet API client."""
    client = APIClient()
    files = client.files(
        site_id=CLOUDNET_SITE,
        date=CLOUDNET_DATE,
        product_id=CLOUDNET_PRODUCT,
    )
    matching_files = [file for file in files if file.filename == CLOUDNET_FILENAME]
    if not matching_files:
        raise FileNotFoundError(
            f"Cloudnet did not return {CLOUDNET_FILENAME} for "
            f"site={CLOUDNET_SITE}, date={CLOUDNET_DATE}, product={CLOUDNET_PRODUCT}"
        )

    target_path.parent.mkdir(parents=True, exist_ok=True)
    downloaded_paths = client.download(
        matching_files[0],
        output_directory=target_path.parent,
        progress=True,
        validate_checksum=True,
    )
    downloaded_path = Path(downloaded_paths[0])
    if downloaded_path != target_path:
        downloaded_path.replace(target_path)
    return target_path


def run_ceilometer_ablh_workbench(
    cloudnet_path: Path,
    raw_path: Path,
    output_dir: Path,
    *,
    variable: str = "beta_smooth",
    time_frequency: str | None = "5min",
    min_range: float = 500.0,
    max_range: float = 4000.0,
    wct_threshold: float = 0.02,
    wct_width: float = 300.0,
    temporal_threshold: float = 1e-5,
    temporal_window_minutes: float = 10.0,
    snr_threshold = 0.01,
    time_sigma = 1,
    range_sigma = 3,
    parameter = 1
) -> tuple[dict[str, Path], Path]:
    """Run WCT and temporal-variance ABLH detection on Cloudnet ceilometer data."""
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load the processed dataset (which will contain 'beta_raw' if variable='beta_raw')
    """"
    dataset = load_cloudnet_for_ablh(
        input_path,
        variable=variable,
        time_frequency=time_frequency,
    )
    """

    dataset = load_cloudnet_for_ablh(input_path=raw_path, variable='beta_raw')
    cloudnet_dataset = load_cloudnet_for_ablh(input_path=cloudnet_path, variable='beta')

    print("Dataset loaded successfully (after initial load_cloudnet_for_ablh).") # Clarified debug print

    # Implement NaN mask from original cdataset['beta']
    # Load original again to get 'beta' variable with its original dimensions before load_cloudnet_for_ablh processes it.
    original_nan_mask = np.isnan(cloudnet_dataset['beta'])

    # Reindex the original NaN mask to match the coordinates of the processed dataset[variable]
    nan_mask_reindexed = original_nan_mask.reindex_like(dataset[variable], method='nearest')

    # Apply the reindexed NaN mask to filter dataset[variable]
    dataset[variable] = dataset[variable].where(~nan_mask_reindexed)

    if dataset[variable].isnull().all():
        print(f"WARNING: After applying cdataset['beta'] NaN mask, '{variable}' is entirely NaN. This might cause issues downstream.")
    else:
        print(f"'{variable}' masked by cdataset['beta'] NaN mask. Valid data points remain.")

    # --- Removed SNR filtering section entirely as per user's request ---

    frequency_label = time_frequency or "native"
    range_label = f"r{min_range:g}-{max_range:g}m"
    output_paths: dict[str, Path] = {}
    results: dict[str, xr.Dataset] = {}

    method_options = {
        "wct": {
            "threshold": wct_threshold,
            "wct_width": wct_width,
            "time_window_minutes": temporal_window_minutes,
        },
        "temporal_variance": {
            "threshold": temporal_threshold,
            "wct_width": wct_width,
            "time_window_minutes":temporal_window_minutes
        },
    }

    print("Starting ABLH detection loop.") # Debug print
    for method in METHODS:
        # Check if dataset[variable] has any finite values before calling detect_ablh
        if not np.isfinite(dataset[variable]).any():
            print(f"Skipping {method} method: '{variable}' has no finite values after filtering.")
            continue # Skip this method if no valid data

        # Original code calls detect_ablh three times. I will just store the result once and reuse.
        ablh_result = detect_ablh(
            dataset,
            input_kind="cloudnet",
            variable=variable,
            method=method,
            min_range=min_range,
            max_range=max_range,
            **method_options[method],
        )
        ablh_result.attrs.update(
            {
                "source_cloudnet_file": str(input_path),
                "cloudnet_variable": variable,
                "time_frequency": frequency_label,
                "wct_threshold": float(wct_threshold),
                "wct_width_m": float(wct_width),
                "temporal_threshold": float(temporal_threshold),
                "temporal_window_minutes": float(temporal_window_minutes),
            }
        )
        output_path = (
            output_dir
            / f"{parameter}_{method}_{input_path.stem}_{variable}_{frequency_label}_{range_label}_ablh.nc"
        )
        ablh_result.to_netcdf(output_path)
        output_paths[method] = output_path
        results[method] = ablh_result
        print(f"ABLH detection completed for method: {method}") # Debug print

    quicklook_path = (
        output_dir
        / f"quicklook_{parameter}_{input_path.stem}_{variable}_{frequency_label}_{range_label}_ablh.png"
    )
    print("Attempting to plot quicklook.") # Debug print
    try:
        plot_cloudnet_ablh_quicklook(
            cloudnet=dataset,
            ablh_results=results,
            variable=variable,
            min_range=min_range,
            max_range=max_range,
            output_path=quicklook_path,
        )
        print("Quicklook plotted successfully.") # Debug print
    except ValueError as e:
        print(f"ERROR: Failed to plot quicklook: {e}")
        # If plotting fails, we should still return the ablh_paths if they were created
        # Or, decide to raise the error further if quicklook is mandatory.
        # For now, just print and continue to return what's available.

    print("run_ceilometer_ablh_workbench finished.") # Debug print
    return output_paths, quicklook_path


def plot_cloudnet_ablh_quicklook(
    *,
    cloudnet: xr.Dataset,
    ablh_results: dict[str, xr.Dataset],
    variable: str,
    min_range: float,
    max_range: float,
    output_path: Path,
) -> None:
    """Plot Cloudnet signal and overlaid ABLH from all requested methods."""
    backscatter = cloudnet[variable].transpose("time", "range")
    ranges = backscatter["range"].values.astype(float)
    range_mask = (ranges >= min_range) & (ranges <= max_range)
    times = backscatter["time"].values
    plot_signal = _log10_positive(backscatter.values[:, range_mask])

    fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)
    mesh = ax.pcolormesh(
        times,
        ranges[range_mask] / 1000.0,
        plot_signal.T,
        shading="auto",
        cmap="viridis",
    )
    colors = {"wct": "white", "temporal_variance": "tab:red"}
    labels = {
        "wct": "ABLH WCT",
        "temporal_variance": "ABLH temporal variance",
    }
    for method, result in ablh_results.items():
        ax.plot(
            result["time"].values,
            result["ablh"].values / 1000.0,
            color=colors.get(method, "tab:orange"),
            lw=1.6 if method == "temporal_variance" else 2.0,
            label=labels.get(method, method),
        )

    title = cloudnet.attrs.get("title", "Cloudnet ceilometer")
    ax.set_title(f"{title}: ABLH from {variable}")
    ax.set_xlabel("UTC time")
    ax.set_ylabel("Range above instrument [km]")
    ax.set_ylim(min_range / 1000.0, max_range / 1000.0)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper right")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    cbar = fig.colorbar(mesh, ax=ax)
    cbar.set_label(f"log10({variable})")
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def _log10_positive(values: np.ndarray) -> np.ndarray:
    # Replace non-positive values with NaN. Original NaNs are preserved.
    processed_values = np.where(values > 0, values, np.nan)

    # Identify finite positive values to calculate the floor.
    finite_positive_mask = np.isfinite(processed_values)

    if not finite_positive_mask.any():
        # If there are no finite positive values, return an array of NaNs.
        # This will result in a blank area on the plot.
        return np.full_like(values, np.nan, dtype=float)

    # Calculate the 1st percentile from the *finite positive* values.
    floor = np.nanpercentile(processed_values[finite_positive_mask], 1)

    # Ensure the floor is positive and finite.
    if not np.isfinite(floor) or floor <= 0:
        floor = np.nanmin(processed_values[finite_positive_mask])

    # Apply the floor only to values that are valid but less than the floor,
    # ensuring NaNs are preserved as NaNs by np.maximum behavior.
    final_values = np.maximum(processed_values, floor)

    return np.log10(final_values)

In [ ]:
def ensure_cloudnet_input(
    input_path: Path,
    *,
    download: bool,
) -> Path:
    """Return an existing Cloudnet file, downloading the default fixture if needed."""
    input_path = input_path.expanduser().resolve()
    if input_path.exists():
        print(f"File already exists: {input_path}")
        return input_path
    if not download:
        raise FileNotFoundError(
            f"Cloudnet input file not found: {input_path}. "
            "Re-run without --no-download or pass --input to an existing file."
        )
    if input_path.name != CLOUDNET_FILENAME:
        raise FileNotFoundError(
            f"Cloudnet input file not found: {input_path}. "
            "Automatic download is only configured for the default Granada CHM15k file."
        )
    print(f"Downloading {CLOUDNET_FILENAME}...")
    return download_cloudnet_lidar_product(target_path)


def download_cloudnet_lidar_product(target_path: Path) -> Path:
    """Download the default Granada CHM15k lidar product with Cloudnet API client."""
    client = APIClient()
    files = client.files(
        site_id=CLOUDNET_SITE,
        date=CLOUDNET_DATE,
        product_id=CLOUDNET_PRODUCT,
    )
    matching_files = [file for file in files if file.filename == CLOUDNET_FILENAME]
    if not matching_files:
        raise FileNotFoundError(
            f"Cloudnet did not return {CLOUDNET_FILENAME} for "
            f"site={CLOUDNET_SITE}, date={CLOUDNET_DATE}, product={CLOUDNET_PRODUCT}"
        )

    target_path.parent.mkdir(parents=True, exist_ok=True)
    downloaded_paths = client.download(
        matching_files[0],
        output_directory=target_path.parent,
        progress=True,
        validate_checksum=True,
    )
    downloaded_path = Path(downloaded_paths[0])
    if downloaded_path != target_path:
        downloaded_path.replace(target_path)
    return target_path


def run_ceilometer_ablh_workbench(
    cloudnet_path: Path,
    raw_path: Path,
    output_dir: Path,
    *,
    variable: str = "beta_smooth",
    time_frequency: str | None = "5min",
    min_range: float = 500.0,
    max_range: float = 4000.0,
    wct_threshold: float = 0.02,
    wct_width: float = 300.0,
    temporal_threshold: float = 1e-5,
    temporal_window_minutes: float = 10.0,
    snr_threshold = 0.01,
    time_sigma = 1,
    range_sigma = 3,
    parameter = 1
) -> tuple[dict[str, Path], Path]:
    """Run WCT and temporal-variance ABLH detection on Cloudnet ceilometer data."""
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load the processed dataset (which will contain 'beta_raw' if variable='beta_raw')
    """"
    dataset = load_cloudnet_for_ablh(
        input_path,
        variable=variable,
        time_frequency=time_frequency,
    )
    """

    dataset = load_cloudnet_for_ablh(input_path=raw_path, variable='beta_raw')
    cloudnet_dataset = load_cloudnet_for_ablh(input_path=cloudnet_path, variable='beta')

    print("Dataset loaded successfully (after initial load_cloudnet_for_ablh).") # Clarified debug print

    # Implement NaN mask from original cdataset['beta']
    # Load original again to get 'beta' variable with its original dimensions before load_cloudnet_for_ablh processes it.
    original_nan_mask = np.isnan(cloudnet_dataset['beta'])

    # Reindex the original NaN mask to match the coordinates of the processed dataset[variable]
    nan_mask_reindexed = original_nan_mask.reindex_like(dataset[variable], method='nearest')

    # Apply the reindexed NaN mask to filter dataset[variable]
    dataset[variable] = dataset[variable].where(~nan_mask_reindexed)

    if dataset[variable].isnull().all():
        print(f"WARNING: After applying cdataset['beta'] NaN mask, '{variable}' is entirely NaN. This might cause issues downstream.")
    else:
        print(f"'{variable}' masked by cdataset['beta'] NaN mask. Valid data points remain.")

    # --- Removed SNR filtering section entirely as per user's request ---

    frequency_label = time_frequency or "native"
    range_label = f"r{min_range:g}-{max_range:g}m"
    output_paths: dict[str, Path] = {}
    results: dict[str, xr.Dataset] = {}

    method_options = {
        "wct": {
            "threshold": wct_threshold,
            "wct_width": wct_width,
            "time_window_minutes": temporal_window_minutes,
        },
        "temporal_variance": {
            "threshold": temporal_threshold,
            "wct_width": wct_width,
            "time_window_minutes":temporal_window_minutes
        },
    }

    print("Starting ABLH detection loop.") # Debug print
    for method in METHODS:
        # Check if dataset[variable] has any finite values before calling detect_ablh
        if not np.isfinite(dataset[variable]).any():
            print(f"Skipping {method} method: '{variable}' has no finite values after filtering.")
            continue # Skip this method if no valid data

        # Original code calls detect_ablh three times. I will just store the result once and reuse.
        ablh_result = detect_ablh(
            dataset,
            input_kind="cloudnet",
            variable=variable,
            method=method,
            min_range=min_range,
            max_range=max_range,
            **method_options[method],
        )
        ablh_result.attrs.update(
            {
                "source_cloudnet_file": str(input_path),
                "cloudnet_variable": variable,
                "time_frequency": frequency_label,
                "wct_threshold": float(wct_threshold),
                "wct_width_m": float(wct_width),
                "temporal_threshold": float(temporal_threshold),
                "temporal_window_minutes": float(temporal_window_minutes),
            }
        )
        output_path = (
            output_dir
            / f"{parameter}_{method}_{input_path.stem}_{variable}_{frequency_label}_{range_label}_ablh.nc"
        )
        ablh_result.to_netcdf(output_path)
        output_paths[method] = output_path
        results[method] = ablh_result
        print(f"ABLH detection completed for method: {method}") # Debug print

    quicklook_path = (
        output_dir
        / f"quicklook_{parameter}_{input_path.stem}_{variable}_{frequency_label}_{range_label}_ablh.png"
    )
    print("Attempting to plot quicklook.") # Debug print
    try:
        plot_cloudnet_ablh_quicklook(
            cloudnet=dataset,
            ablh_results=results,
            variable=variable,
            min_range=min_range,
            max_range=max_range,
            output_path=quicklook_path,
        )
        print("Quicklook plotted successfully.") # Debug print
    except ValueError as e:
        print(f"ERROR: Failed to plot quicklook: {e}")
        # If plotting fails, we should still return the ablh_paths if they were created
        # Or, decide to raise the error further if quicklook is mandatory.
        # For now, just print and continue to return what's available.

    print("run_ceilometer_ablh_workbench finished.") # Debug print
    return output_paths, quicklook_path


def plot_cloudnet_ablh_quicklook(
    *,
    cloudnet: xr.Dataset,
    ablh_results: dict[str, xr.Dataset],
    variable: str,
    min_range: float,
    max_range: float,
    output_path: Path,
) -> None:
    """Plot Cloudnet signal and overlaid ABLH from all requested methods."""
    backscatter = cloudnet[variable].transpose("time", "range")
    ranges = backscatter["range"].values.astype(float)
    range_mask = (ranges >= min_range) & (ranges <= max_range)
    times = backscatter["time"].values
    plot_signal = _log10_positive(backscatter.values[:, range_mask])

    fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)
    mesh = ax.pcolormesh(
        times,
        ranges[range_mask] / 1000.0,
        plot_signal.T,
        shading="auto",
        cmap="viridis",
    )
    colors = {"wct": "white", "temporal_variance": "tab:red"}
    labels = {
        "wct": "ABLH WCT",
        "temporal_variance": "ABLH temporal variance",
    }
    for method, result in ablh_results.items():
        ax.plot(
            result["time"].values,
            result["ablh"].values / 1000.0,
            color=colors.get(method, "tab:orange"),
            lw=1.6 if method == "temporal_variance" else 2.0,
            label=labels.get(method, method),
        )

    title = cloudnet.attrs.get("title", "Cloudnet ceilometer")
    ax.set_title(f"{title}: ABLH from {variable}")
    ax.set_xlabel("UTC time")
    ax.set_ylabel("Range above instrument [km]")
    ax.set_ylim(min_range / 1000.0, max_range / 1000.0)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper right")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    cbar = fig.colorbar(mesh, ax=ax)
    cbar.set_label(f"log10({variable})")
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def _log10_positive(values: np.ndarray) -> np.ndarray:
    # Replace non-positive values with NaN. Original NaNs are preserved.
    processed_values = np.where(values > 0, values, np.nan)

    # Identify finite positive values to calculate the floor.
    finite_positive_mask = np.isfinite(processed_values)

    if not finite_positive_mask.any():
        # If there are no finite positive values, return an array of NaNs.
        # This will result in a blank area on the plot.
        return np.full_like(values, np.nan, dtype=float)

    # Calculate the 1st percentile from the *finite positive* values.
    floor = np.nanpercentile(processed_values[finite_positive_mask], 1)

    # Ensure the floor is positive and finite.
    if not np.isfinite(floor) or floor <= 0:
        floor = np.nanmin(processed_values[finite_positive_mask])

    # Apply the floor only to values that are valid but less than the floor,
    # ensuring NaNs are preserved as NaNs by np.maximum behavior.
    final_values = np.maximum(processed_values, floor)

    return np.log10(final_values)

In [ ]:
from scipy.ndimage import gaussian_filter
import numpy as np
import xarray as xr # Import xarray for the example to work

def calculate_gaussian_snr(
    dataset: xr.Dataset,
    variable: str,
    time_sigma: float = 1.0,  # Standard deviation in array indices for time dimension
    range_sigma: float = 5.0 # Standard deviation in array indices for range dimension
) -> xr.DataArray:
    """
    Calculates SNR for a given variable in an xarray Dataset using a Gaussian filter
    to estimate mean and standard deviation within a sliding window.

    Args:
        dataset (xr.Dataset): The input xarray Dataset.
        variable (str): The name of the variable to calculate SNR for (e.g., 'beta_raw').
        time_sigma (float): Standard deviation (in array indices) for the Gaussian kernel
                            along the time dimension.
        range_sigma (float): Standard deviation (in array indices) for the Gaussian kernel
                             along the range dimension.

    Returns:
        xr.DataArray: A new DataArray containing the calculated SNR.
    """
    if variable not in dataset.data_vars:
        raise ValueError(f"Variable '{variable}' not found in the dataset.")

    data_array = dataset[variable].transpose('time', 'range')

    # Convert non-finite values to NaN for robust statistical calculation
    data_values = data_array.values.astype(float)
    nan_mask = np.isnan(data_values) | np.isinf(data_values)

    # Apply Gaussian filter to calculate smoothed mean
    # Use a copy where NaNs are handled, e.g., by filling them or masking during filter
    # For simplicity, we'll just filter the original data and deal with NaNs later
    # A more sophisticated approach would involve `scipy.ndimage.gaussian_filter` with mask handling.

    # Calculate smoothed mean, treating NaNs as zeros (or interpolating, but simpler here)
    # And then normalizing by a smoothed mask to get a true mean
    # For direct application, we might fill NaNs for the filter, then reintroduce them.
    # A common approach for Gaussian filtering with NaNs is to fill NaNs, filter, then restore.
    # For demonstration, we'll use a simpler approach which might be less accurate near NaNs.

    # Replace NaNs temporarily for gaussian_filter, which doesn't directly handle them well.
    # A robust way is to compute sum and count separately with Gaussian kernel.
    # Simplified approach for this example:
    filled_data = np.nan_to_num(data_values, nan=0.0)

    # Calculate smoothed sum and count (weight)
    gaussian_sum = gaussian_filter(filled_data, sigma=(time_sigma, range_sigma), mode='constant', cval=0.0)
    # Create a mask for valid data points (1 for valid, 0 for nan/inf)
    valid_mask = (~nan_mask).astype(float)
    gaussian_count = gaussian_filter(valid_mask, sigma=(time_sigma, range_sigma), mode='constant', cval=0.0)

    # Avoid division by zero
    smoothed_mean = np.divide(gaussian_sum, gaussian_count, out=np.zeros_like(gaussian_sum), where=gaussian_count != 0)

    # Calculate smoothed standard deviation
    # E[(X - E[X])^2] = E[X^2] - (E[X])^2
    # Calculate smoothed E[X^2]
    squared_data_filled = np.nan_to_num(data_values**2, nan=0.0)
    gaussian_sum_sq = gaussian_filter(squared_data_filled, sigma=(time_sigma, range_sigma), mode='constant', cval=0.0)
    smoothed_mean_sq = np.divide(gaussian_sum_sq, gaussian_count, out=np.zeros_like(gaussian_sum_sq), where=gaussian_count != 0)

    smoothed_variance = smoothed_mean_sq - smoothed_mean**2
    smoothed_variance[smoothed_variance < 0] = 0 # Ensure variance is non-negative
    smoothed_std = np.sqrt(smoothed_variance)

    # Calculate SNR = smoothed_mean / smoothed_std
    # Handle cases where std is zero to avoid division by zero
    snr_values = np.divide(smoothed_mean, smoothed_std, out=np.zeros_like(smoothed_mean), where=smoothed_std != 0)

    # Create a new xarray DataArray for SNR with original coordinates
    snr_array = xr.DataArray(
        snr_values,
        coords=data_array.coords,
        dims=data_array.dims,
        name=f"snr_{variable}",
        attrs={'long_name': f'SNR of {variable} (Gaussian smoothed)', 'units': 'dimensionless'}
    )

    # Mask out regions where there was no valid data to begin with
    snr_array = snr_array.where(~nan_mask)

    return snr_array


In [ ]:
def load_cloudnet_for_ablh(
    input_path: Path,
    *,
    variable: str,
    time_frequency: str | None = None,
) -> xr.Dataset:
    """Load a Cloudnet lidar product and keep only fields needed by ABLH."""
    dataset = xr.open_dataset(input_path)
    if variable not in dataset:
        available = ", ".join(dataset.data_vars)
        raise KeyError(
            f"Variable '{variable}' not found. Available variables: {available}"
        )
    # Sort the dataset by time to ensure monotonicity for resampling
    dataset = dataset.sortby('time')

    # Handle potential duplicate time points before resampling
    # Group by time and take the median to ensure a unique and monotonic time index
    if not dataset.indexes['time'].is_unique:
        dataset = dataset.groupby('time').median()

    backscatter = dataset[variable].transpose("time", "range")
    if time_frequency:
        backscatter = backscatter.resample(time=time_frequency).median(skipna=True)

    output = xr.Dataset({variable: backscatter})
    if "height" in dataset:
        output["height"] = dataset["height"]
    output.attrs.update(dataset.attrs)
    return output

In [ ]:

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT)
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT_DIR)
    parser.add_argument(
        "--no-download",
        action="store_true",
        help="Fail if the default Cloudnet file is missing instead of downloading it.",
    )
    parser.add_argument(
        "--variable",
        default="beta_raw",
        choices=["beta_smooth", "beta", "beta_raw"],
        help="Cloudnet backscatter variable used for detection.",
    )
    parser.add_argument(
        "--time-frequency",
        default='5min',
        help="Pandas/xarray resampling frequency. Use 'none' for native time.",
    )
    parser.add_argument("--min-range", type=float, default=500.0)
    parser.add_argument("--max-range", type=float, default=4000.0)
    parser.add_argument("--wct-threshold", type=float, default=0.05)
    parser.add_argument("--wct-width", type=float, default=300.0)
    parser.add_argument("--temporal-threshold", type=float, default=1e-5)
    parser.add_argument("--temporal-window-minutes", type=float, default=15.0)
    return parser.parse_args([])

def main() -> None:
    args = parse_args()
    input_path = ensure_cloudnet_input(args.input, download=not args.no_download)
    time_frequency = (
        None if args.time_frequency.lower() == "none" else args.time_frequency
    )
    ablh_paths, quicklook_path = run_ceilometer_ablh_workbench(
        input_path=input_path,
        output_dir=args.output_dir,
        variable=args.variable,
        time_frequency=time_frequency,
        min_range=args.min_range,
        max_range=args.max_range,
        wct_threshold=args.wct_threshold,
        wct_width=args.wct_width,
        temporal_threshold=args.temporal_threshold,
        temporal_window_minutes=args.temporal_window_minutes,
        parameter=f'{args.wct_width}_{args.wct_threshold}' #just for the quicklook file name
    )
    for method, path in ablh_paths.items():
        print(f"Saved {method} ABLH NetCDF: {path}")
    print(f"Saved ABLH quicklook: {quicklook_path}")


if __name__ == "__main__":
    main()

In [ ]:
# Assuming DEFAULT_INPUT, DEFAULT_OUTPUT_DIR, ensure_cloudnet_input, run_ceilometer_ablh_workbench
# are defined in preceding cells (QII8cft87THc and Cafwgldzpb_7).

input_path = ensure_cloudnet_input(DEFAULT_INPUT, download=True)
output_dir = DEFAULT_OUTPUT_DIR

variable = "beta_raw"
time_frequency = "5min"
min_range = 500.0
max_range = 4000.0
wct_threshold = 0.05 # Consistent with common usage and user comments
wct_width = 300.0 # Consistent with default and user comments
temporal_threshold = 1e-5
temporal_window_minutes = 10.0 # Consistent with run_ceilometer_ablh_workbench default
snr_threshold = 0 # New parameter for SNR filtering
time_sigma = 1 # New parameter for Gaussian SNR
range_sigma = 2 # New parameter for Gaussian SNR
parameter = f'{wct_width}_{wct_threshold}' # For quicklook file name

ablh_paths, quicklook_path = run_ceilometer_ablh_workbench(
      input_path=input_path,
      output_dir=output_dir,
      variable=variable,
      time_frequency=time_frequency,
      min_range=min_range,
      max_range=max_range,
      wct_threshold=wct_threshold,
      wct_width=wct_width,
      temporal_threshold=temporal_threshold,
      temporal_window_minutes=temporal_window_minutes,
      snr_threshold=snr_threshold, # Pass new argument
      time_sigma=time_sigma, # Pass new argument
      range_sigma=range_sigma, # Pass new argument
      parameter=0
  )

print(f"ABLH paths: {ablh_paths}")
print(f"Quicklook path: {quicklook_path}")

In [ ]:
# Assuming DEFAULT_INPUT, DEFAULT_OUTPUT_DIR, ensure_cloudnet_input, run_ceilometer_ablh_workbench
# are defined in preceding cells (QII8cft87THc and Cafwgldzpb_7).

variable = "beta_raw"
time_frequency = "5min"
min_range = 500.0
max_range = 4000.0
wct_threshold = 0.05 # Consistent with common usage and user comments
wct_width = 300.0 # Consistent with default and user comments
temporal_threshold = 1e-5
temporal_window_minutes = 10.0 # Consistent with run_ceilometer_ablh_workbench default
snr_threshold = 2.5 # New parameter for SNR filtering
time_sigma = 1 # New parameter for Gaussian SNR
range_sigma = 2 # New parameter for Gaussian SNR

loop_array = [.5, 1, 1.5, 2, 2.5, 3] #time_sigma
loop_array = [1]
for var_loop in loop_array:
  ablh_paths, quicklook_path = run_ceilometer_ablh_workbench(
        raw_path=RAW_INPUT,
        cloudnet_path = CLOUDNET_INPUT,
        output_dir=DEFAULT_OUTPUT_DIR,
        variable=variable,
        time_frequency=time_frequency,
        min_range=min_range,
        max_range=max_range,
        wct_threshold=wct_threshold,
        wct_width=wct_width,
        temporal_threshold=temporal_threshold,
        temporal_window_minutes=temporal_window_minutes,
        snr_threshold=snr_threshold, # Pass new argument
        time_sigma=var_loop, #time_sigma, # Pass new argument
        range_sigma=range_sigma, # Pass new argument
        parameter=f'time_sigma_{var_loop}' #for differentiating file name
    )

print(f"ABLH paths: {ablh_paths}")
print(f"Quicklook path: {quicklook_path}")

In [ ]:
ceilo = xr.open_dataset(r'/content/20240207_Granada_CHM170119_000.nc')
ceilo

In [ ]:
ceilo.time

In [ ]:
filepath = Path(r'/content/20240207_Granada_CHM170119_000.nc')
filepath.exists()
new_var, file_ = run_ceilometer_ablh_workbench(input_path=filepath, output_dir=Path.cwd(), variable='beta_raw', wct_threshold=0.05)
new_var

In [ ]:
new_var['wct']

7/1/2026: testing gaussian snr calculation:

to start, testing the values (0.01, 0.05, 0.1, 0.5, 1, 3, 5, 10) on each of the snr_threshold, time_sigma, and range_sigma variables. Range_sigma should stay at .5 or above, .1 or below everything is eliminated.

Seriously testing snr_threshold first because it has the biggest impact on the data. Looks best between 1-5, testing again with the values (1.5, 2, 2.5, 3.5, 4, 4.5). I'm going with 2.5 for now, up to changing though.

Testing time_sigma for real now. I think I'm going with 1.0 for that, it's a trade off between smoothing and data loss. I think I'm going to be cycling through each variable a few times.

Testing range_sigma for real now. Not a big fan of any of the options - switching time_sigma to .1 to try and get some data back and trying again. Ok, testing more values between 1 and 5 again, because it looks best in there somewhere. Maybe going with range_sigma of 2? I feel like everything after that leaves too many holes.

Testing time_sigma from .5 to 3 again (.5, 1, 1.5, 2, 2.5, 3). It's a double-edged sword still - .5 is best to prevent data loss, but in response it doesn't do much to prevent noise.

6/29/2026: more wct sensitivity testing, and maybe figuring out preprocessing?

trying wct threshold on other days; 2024-02-06 results:

Trial #|wct_width|wct_threshold|stddev (15:00-18:00)|all-day stddev
-|-|-|-|-
1|350|.02|
2|350|.03|
3|350|.04|
4|350|.05|
5|350|.06|
6|350|.07|
7|400|.05|
8|500|.05|

Not following through because there's a big cloud that day that's impossible. Switching focus to cleaning up the data again, the calculate_gaussian_snr function and its implementation in the run_ceilometer_ablh_workbench function are there to screen out any noise (and hopefully clouds) from the dataset. That system has 3 relevant variables that I need to tinker with now: snr_threshold, time_sigma and range_sigma (sigma in the sense of std dev in a normal distribution)

6/26/2026: Preprocessing the raw icenet files:

I'm pretty sure that the `lidarpy.preprocessing.preprocess(file_path)` contains what I need, but I can't figure out where or how to implement it for it to actually work. Something to bring up next time I see him. I might go into sensitivity testing for the WCT method at this point, since that seems less affected by these data format issues. Also, I can't figure out how to do standard deviation on the icenet files? something to bring up as well. Interestingly there is a small difference between the beta_raw/5min between cloudnet and icenet small difference of cloudnet being slightly smoother.

WCT testing for 2024-02-07 (done on cloudnet beta_raw/5min files b/c I can't get standard deviation working on the icenet files.):
Trial #|wct_width|wct_threshold|stddev (15:00-18:00)|all-day stddev
-|-|-|-|-
1|300|.05|313.20574951
2|200|.05|289.9677124
3|100|.05|362.68008423
4|400|.05|310.23028564
5|500|.05|323.9772644
6|250|.05|302.35092163
7|350|.05|308.63421631|231.32591248
8|350|.04|299.84030151
9|350|.03|291.29067993
10|350|.06|307.04949951|238.16430664
11|350|.07|290.13214111|246.47663879
12|350|.08|245.08659363

Visually I think my favorites are 350, 300, 400, in that order; also, I'm somewhere between .05-.07. Don't let the sttdev fool you, .08 won't work (big spikes from 12-14:00). I might check .05-.07 on a few other days just to see


6/24/2026: figuring out what the deal is with beta_smooth and time_frequency. All results are discussing the temporal variance method. Detailed documentation at the bottom underneath the std cell. Using the date 2024-02-07

test 1: 10/15/20 minutes for both time_frequency = '5min' and ='none'

results: there is a small difference between 10/15/20 minutes now on 'none' time frequency. It's also a bit smoother in places but with much worse spikes. Not convinced this is better.

test 2: 10/15/20 minutes on beta, time frequency back to 5min.

results: background lidar data looks notably different. not much of a difference between 10/15 but 20 is smoother in some parts.

test 3: 10/15/20 minutes on beta_raw, time frequency still on 5min

results: beta_raw data is unusable. As I should have expected, clearly unscreened. WCT still works fairly well, but temporal variance spikes from the bottom of the graph to the top.

test 4: 10/15/20 minutes on beta, time frequency on 'none'.

results: I'd say it does look better than beta_smooth native, but still looks pretty terrible. Parts of it are kinda smooth actually, and there are fewer egregious spikes, but still really hard to read in general. 10/15/20 all have noticeable differences between them.

test 5: every above setting, but with temporal window of 5mins

results: (compared to 10min) beta|5min looks about the same, beta|native looks worse, beta_raw|5min looks about the same, beta_smooth|5min looks about the same as well, and beta_smooth|native looks better in some spots, but still has bad spikes like the rest.

test 6: a handful of the above settings, but on the icenet data from the folder

results: data from the folder he pointed out is clearly formatted in a different way, I think it might not have been preprocessed at all? Can only figure out how to run beta_raw|none on it, and that produced a complete spiderweb-type mess.

2026-06-10 Data trials
Trial #|WCT threshold|WCT width|temporal threshold|temporal window minutes
------|------------|---------|-----------------|-
1|.02|300|1e-5|10
2|.03|300|1e-5|10
3|.04|''|''|''
4|.05|''|''|''
5|.01|''|''|''
6|.06|''|''|''
7|.02|''|2e-5|''
8|''|''|.0001|''
9|''|''|.001|''
10|''|''|.01|''
11|''|''|.1|''
12|''|''|1e-6|''
13|''|''|1e-5|15
14|''|''|''|30
15|''|''|''|20
16|''|200|''|10
17|''|100|''|''
18|''|400|''|''
19|''|500|''|''
20|''|50|''|''
21|.05|100|1e-5|20

Conclusions:

changing temporal threshold has no visible effect, except at high values (closer to .1) the red line progressively disappears.

Changing wct threshold has an effect on the white line but none on the red line. Gets closer to the red line the higher the value.

Changing time interval only seems to change red line: 30 mins is too much, oversimplifies and leaves more gaps; 15 mins is about the same as 10, really jagged. 20 mins might be a decent compromise.

WCT width only affects the white line. Lower values have a similar effect to higher wct threshold values, although less smooth. Definitely no lower than 100, 50 reacts really weird.

Maybe I should try changing these parameters on a day with messier data

2026-06-06 Data Trials
Trial #|WCT threshold|WCT width|temporal threshold|temporal window minutes
------|------------|---------|-----------------|-
1|.02|300|1e-5|10
2|.05|100|1e-5|20
3|.05|100|1e-5|15
4|.05|200|1e-5|15
5|.05|250|1e-4|15
6|.05|300|1e-5|15

trial 2 has a better looking white line and arguably worse red line, although it's still kinda chaos in the final section. Most accurate reading of the final section (assuming it's a dust plume and not meant to be measured) is the white line in the default settings. First 2/3 of the day are actually really accurate though, for red on both and white on 2. (unless white on two is measuring more than just the CBL?)
update: daytime white line for trial 6 looks pretty great actually. Red line appears unsalvageable.


Other Dates:
2026-06-05 (day before) with the same parameters as trial 6 (.05/300/1e-5/15) looks pretty great first try. 2026-06-04 is a pretty messy graph, but given that the graph looks decent during the day especially. 2026-06-02 looks okay in some parts but it's messy and doesn't quite slope up correctly with the cbl in the morning. 2026-05-31 is really messy at night but looks decent during the day. Overall I think I'm really happy with these parameters (especially if we're more worried about maximum per day than anything)! I didn't think that the white line would be more consistent, but it just turned out to be easier to manipulate those parameters I guess. It's much harder to iron the spikes in the red line.

Looking again at their own paper they sent, they say something about "dilation values" being good between 200-450, is that equivalent to WCT width here? Idk. WCT threshold tends to work well between .04 and .08, so that checks out. That study actually also landed on 300 and .05 as the optimal values for WCT, which is what I've landed on! Definitely keeping those values then. I think this means I'm favoring WCT over temporal variance? I just can't adjust the latter with the window variable nearly as well as I can the former with the threshold and width variables. Article also mentioned how it is really difficult to get a good height with dust layers, you can mess with threshold values but it's really unreliable. Additionally, this article: https://upcommons.upc.edu/server/api/core/bitstreams/3316944b-e4cb-4c6a-a138-e61879989af3/content that talks about the temporal variance strategy does use a window of about 15 minutes. There was another article that sounded like it was using a window of 30 mins though. One article I found frequently stated that the variance method on its own, while being powerful, isn't super reliable/consistent/easy to read, so my choice to go with WCT (white line) also makes sense.

In [ ]:
#for reading .nc files
file_path = r"/content/300.0_0.05_wct_20240207_granada_chm15k_77a75f3b_beta_raw_5min_r500-4000m_ablh.nc"
lidar_data = xr.open_dataset(file_path)
print(lidar_data.ablh_height)
lidar_data.close()

In [ ]:
#Goal: get standard deviation for the 3-6pm window (highest ablh for the day being investigated [2024-02-07])
length = 288 #288 for 5min frequency, 5760 for native
file_path = r'/content/350.0_0.08_wct_20240207_granada_chm15k_77a75f3b_beta_raw_5min_r500-4000m_ablh.nc'
lidar_data = xr.open_dataset(file_path)
relevant_data = lidar_data.ablh_height
#relevant_data = lidar_data.ablh_height[int(length*5/8):int((length*3/4)+1)] # 15:00 to 18:00
print(np.std(relevant_data))

Temporal Variance
Trial #|Temporal window minutes|variable|time frequency|standard deviation 15:00 - 18:00
------|------------|---------|-----------------|-
1|10|beta_smooth|5min|207.67591858
2|15|beta_smooth|5min|207.67591858
3|20|beta_smooth|5min|219.02207947
4|10|beta_smooth|none|560.7230835
5|15|beta_smooth|none|640.02832031
6|20|beta_smooth|none|700.55932617
7|10|beta|5min|261.50491333
8|15|beta|5min|261.50491333
9|20|beta|5min|257.56860352
10|10|beta_raw|5min|989.83428955
11|15|beta_raw|5min|989.83428955
12|20|beta_raw|5min|705.08325195
13|10|beta|none|306.1678772
14|15|beta|none|311.23596191
15|20|beta|none|293.55111694
16|5|beta_smooth|5min|207.67591858
17|5|beta_smooth|none|475.6491394
18|5|beta|5min|261.50491333
19|5|beta_raw|5min|989.83428955
20|5|beta|none|329.41998291

WCT (.05/300)

Trial #|variable|time frequency|standard deviation 15:00 - 18:00
------|------------|---------|-----------------
1|beta_smooth|5min|314.16577148
2|beta_smooth|none|298.28344727
3|beta|5min|310.00823975
4|beta_raw|5min|313.20574951
5|beta|none|300.08639526

##Trying to graph multiple variations of a method on one graph

Version of plotting function with more color and label flexibility included

In [ ]:
def plot_cloudnet_ablh_quicklook_multi(
    *,
    cloudnet: xr.Dataset,
    ablh_results: dict[str, xr.Dataset],
    variable: str,
    min_range: float,
    max_range: float,
    output_path: Path,
    colors: dict[str, str],
    labels: dict[str, str]
) -> None:
    """Plot Cloudnet signal and overlaid ABLH from all requested methods."""
    backscatter = cloudnet[variable].transpose("time", "range")
    ranges = backscatter["range"].values.astype(float)
    range_mask = (ranges >= min_range) & (ranges <= max_range)
    times = backscatter["time"].values
    plot_signal = _log10_positive(backscatter.values[:, range_mask])

    fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)
    mesh = ax.pcolormesh(
        times,
        ranges[range_mask] / 1000.0,
        plot_signal.T,
        shading="auto",
        cmap="viridis",
    )
    colors = colors
    labels = labels
    for method, result in ablh_results.items():
        ax.plot(
            result["time"].values,
            result["ablh"].values / 1000.0,
            color=colors.get(method, "tab:orange"),
            lw=1.6 if method == "temporal_variance" else 2.0,
            label=labels.get(method, method),
        )

    title = cloudnet.attrs.get("title", "Cloudnet ceilometer")
    ax.set_title(f"{title}: ABLH from {variable}")
    ax.set_xlabel("UTC time")
    ax.set_ylabel("Range above instrument [km]")
    ax.set_ylim(min_range / 1000.0, max_range / 1000.0)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper right")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    cbar = fig.colorbar(mesh, ax=ax)
    cbar.set_label(f"log10({variable})")
    fig.savefig(output_path, dpi=160)
    plt.close(fig)

plot_cloudnet_ablh_quicklook_multi(
    cloudnet=load_cloudnet_for_ablh(
        input_path=ensure_cloudnet_input(DEFAULT_INPUT, download=True),
        variable='beta_raw',
        time_frequency='5min',
    ),
    ablh_results={'wct1':xr.open_dataset(r"/content/350.0_0.05_wct_20240206_granada_chm15k_77a75f3b_beta_raw_5min_r500-4000m_ablh.nc"),
                  'wct2':xr.open_dataset(r'/content/400.0_0.05_wct_20240206_granada_chm15k_77a75f3b_beta_raw_5min_r500-4000m_ablh.nc'),
                  'wct3':xr.open_dataset(r'/content/500.0_0.05_wct_20240206_granada_chm15k_77a75f3b_beta_raw_5min_r500-4000m_ablh.nc')
                  },
    variable='beta_raw',
    min_range=500,
    max_range=4000,
    output_path=r'/content/testing_quicklook_20240206_3.png',
    colors={'wct1':'white', 'wct2':'red', 'wct3':'blue'},
    labels = {
        "wct1": "ABLH WCT width = 350",
        "wct2": "ABLH WCT width = 400",
        'wct3': "ABLH WCT width = 500"
    },
)

In [ ]:
rdataset = xr.open_dataset(RAW_INPUT)
cdataset = xr.open_dataset(DEFAULT_INPUT)

In [ ]:
rdataset.dims

In [ ]:
cdataset.dims

In [ ]:
cdataset['beta'].plot(x='time')

In [ ]:
nan_mask = np.isnan(cdataset['beta'])
print("NaN mask for cdataset['beta']:")
print(nan_mask.head())


In [ ]:
nan_mask.plot(x='time', y='range')
plt.title("NaN Mask for cdataset['beta']")
plt.xlabel('Time UTC')
plt.ylabel('Range [m]')
plt.show()

In [ ]:
rdataset['beta_raw_masked_by_nan_mask'] = rdataset['beta_raw'].where(~nan_mask)

rdataset['beta_raw_masked_by_nan_mask'].values

In [ ]:
nan_mask_reindexed = nan_mask.reindex_like(rdataset['beta_raw'], method='nearest')
rdataset['beta_raw_masked_reindexed'] = rdataset['beta_raw'].where(~nan_mask_reindexed)

if rdataset['beta_raw_masked_reindexed'].isnull().all():
    print("WARNING: The new variable 'beta_raw_masked_reindexed' is entirely NaN even after reindexing the mask. This suggests that 'rdataset[\'beta_raw\']' itself is mostly NaN in areas where 'cdataset[\'beta\']' has valid data, or that the reindexing introduced NaNs where there was no overlap.")
else:
    print("New variable 'beta_raw_masked_reindexed' created successfully using reindexed mask. Displaying first few values:")
    display(rdataset['beta_raw_masked_reindexed'].head())

In [ ]:
import matplotlib.pyplot as plt

rdataset['beta_raw_masked_reindexed'].sortby('time').plot(x='time', y='range', cmap='jet', vmin=0,vmax=15)
plt.title("rdataset['beta_raw'] Masked by Reindexed cdataset['beta'] NaN Mask")
plt.xlabel('Time UTC')
plt.ylabel('Range [m]')
plt.show()

In [ ]:
print("Dimensions of rdataset['beta_raw']:", rdataset['beta_raw'].dims)
print("Coordinates of rdataset['beta_raw']:", rdataset['beta_raw'].coords.keys())
print("\nDimensions of cdataset['beta']:", cdataset['beta'].dims)
print("Coordinates of cdataset['beta']:", cdataset['beta'].coords.keys())

In [ ]:
filtered_beta_raw = rdataset['beta_raw'].where(~np.isnan(cdataset['beta']))

if filtered_beta_raw.isnull().all():
    print("WARNING: After applying the filter, the resulting 'beta_raw' DataArray contains no numeric data points to plot.")
else:
    filtered_beta_raw.plot(x='time')

In [ ]:
rdataset['beta_raw_masked_by_cbeta'] = rdataset['beta_raw'].where(~np.isnan(cdataset['beta']))

if rdataset['beta_raw_masked_by_cbeta'].isnull().all():
    print("WARNING: The new variable 'beta_raw_masked_by_cbeta' is entirely NaN, meaning no valid data points exist after masking.")
else:
    print("New variable 'beta_raw_masked_by_cbeta' created successfully. Displaying first few values:")
    print(rdataset['beta_raw_masked_by_cbeta'].head())